In [1]:
import numpy as np
import pandas as pd

def generar_personas(n=500, seed=42):
    rng = np.random.default_rng(seed)

    # ---------------------------
    # CIUDADES (Guayaquil > Quito > Cuenca)
    # ---------------------------
    ciudades = np.array([
        "Guayaquil","Quito","Cuenca","Manta","Portoviejo","Ambato","Loja","Machala",
        "Esmeraldas","Ibarra","Riobamba","Santo Domingo","Quevedo","Salinas"
    ])

    probs_ciudad = np.array([0.28, 0.22, 0.14, 0.06, 0.05, 0.05, 0.04, 0.04,
                             0.03, 0.03, 0.025, 0.025, 0.02, 0.02])
    probs_ciudad = probs_ciudad / probs_ciudad.sum()
    ciudad = rng.choice(ciudades, size=n, p=probs_ciudad)

    # Código de provincia (2 dígitos) usado en identificación (cédula-like)
    prov_code = {
        "Guayaquil": "09",        # Guayas
        "Quito": "17",            # Pichincha
        "Cuenca": "01",           # Azuay
        "Manta": "13",            # Manabí
        "Portoviejo": "13",       # Manabí
        "Ambato": "18",           # Tungurahua
        "Loja": "11",             # Loja
        "Machala": "07",          # El Oro
        "Esmeraldas": "08",       # Esmeraldas
        "Ibarra": "10",           # Imbabura
        "Riobamba": "06",         # Chimborazo
        "Santo Domingo": "23",    # Santo Domingo de los Tsáchilas
        "Quevedo": "12",          # Los Ríos
        "Salinas": "24",          # Santa Elena
    }

    # ---------------------------
    # SEXO (case inconsistente)
    # ---------------------------
    sexo_base = rng.choice(["M", "F"], size=n, p=[0.52, 0.48])
    sexo = np.where(rng.random(n) < 0.5, sexo_base, np.char.lower(sexo_base))
    is_male = np.char.upper(sexo.astype(str)) == "M"

    # ---------------------------
    # EDAD (20–63, media ~35, sesgo ligero, 3% NA)
    # ---------------------------
    edad = rng.gamma(shape=8.5, scale=2.2, size=n) + 18
    edad = np.clip(np.round(edad), 20, 63).astype(float)
    edad[rng.random(n) < 0.03] = np.nan

    # ---------------------------
    # ALTURA (cm) y PESO (lb) (hombres mayor en promedio)
    # ---------------------------
    altura_cm = np.zeros(n)
    altura_cm[is_male] = rng.normal(loc=176, scale=7.0, size=is_male.sum())
    altura_cm[~is_male] = rng.normal(loc=165, scale=6.5, size=(~is_male).sum())
    altura_cm = np.clip(altura_cm, 145, 205)
    altura_cm = np.round(altura_cm, 1)

    peso_lb = (altura_cm - 100) * 2.2 + rng.normal(0, 12, size=n)
    peso_lb[is_male] += rng.normal(8, 6, size=is_male.sum())
    peso_lb[~is_male] -= rng.normal(4, 5, size=(~is_male).sum())
    peso_lb = np.clip(peso_lb, 95, 320)
    peso_lb = np.round(peso_lb, 1)

    # ---------------------------
    # NOMBRES (1 nombre + 2 apellidos) (más variedad)
    # ---------------------------
    nombres = np.array([
        "Ana","María","Jose","Juan","Luis","Carmen","Diana","Pedro","Sofía","Diego",
        "Camila","Andrés","Valeria","Miguel","Paola","Daniel","Carla","Jorge",
        "Fernanda","Ricardo","Elena","Sebastián","Isabela","Gustavo","Lucía",
        "Mateo","Adriana","Esteban"," Natalia","Kevin","Gabriela","Cristian",
        "Alejandra","Samuel","Andrea","Bryan","Tatiana","Javier","Melanie",
        "Carlos","Vanessa","Alejandro","Nicole","David","Karen"
    ])

    apellidos = np.array([
        "García","Rodriguez","Moreno","Vera","Mendoza","Paredes","Sanchez","Zambrano",
        "Cedeño","Loor","Cruz","Ortega","Molina","Bravo","Rojas","Torres",
        "Castro","Flores","Guerrero","Jiménez","Peña","Villacís","Acosta",
        "Chávez","Reyes","Alvarado","Carrillo","Salazar","Espinoza","López",
        "Benítez","Romero","Herrera","Silva","Valencia","Suárez","Tapia",
        "Arévalo","Navarro","Villamar","Delgado","Cornejo","Quintero","Mera","Solís"
    ])

    # 1 nombre + 2 apellidos
    nombre_raw = (
        rng.choice(nombres, size=n) + " " +
        rng.choice(apellidos, size=n) + " " +
        rng.choice(apellidos, size=n)
    )

    # Variaciones de case
    r = rng.random(n)
    nombre = np.where(r < 0.25, np.char.upper(nombre_raw),
             np.where(r < 0.50, np.char.lower(nombre_raw),
             np.where(r < 0.75, pd.Series(nombre_raw).str.title().to_numpy(),
                      nombre_raw)))

    # Ensuciamos con espacios extra y dobles espacios (para limpieza)
    nombre = pd.Series(nombre)
    nombre = nombre.where(rng.random(n) > 0.12, " " + nombre + " ")
    nombre = nombre.where(rng.random(n) > 0.10, nombre.str.replace(" ", "  ", regex=False))
    nombre = nombre.to_numpy(dtype=object)

    # ---------------------------
    # EDUCACIÓN (limpio, pero correlacionado con salario)
    # ---------------------------
    niveles = np.array(["Primaria", "Secundaria", "Tecnico", "Universidad", "Maestria"])
    p_edu = np.array([0.06, 0.30, 0.18, 0.36, 0.10])
    nivel_educacion = rng.choice(niveles, size=n, p=p_edu)

    # ---------------------------
    # SALARIO (sesgo derecha, min 480, mean ~766, max 5000)
    # ---------------------------
    edu_multiplier = {
        "Primaria": 0.92,
        "Secundaria": 1.00,
        "Tecnico": 1.08,
        "Universidad": 1.22,
        "Maestria": 1.55
    }

    base = rng.lognormal(mean=np.log(650), sigma=0.55, size=n)  # fuerte sesgo derecha
    mult = np.vectorize(edu_multiplier.get)(nivel_educacion)

    city_bonus = np.where(np.isin(ciudad, ["Quito", "Guayaquil"]), 1.05,
                  np.where(ciudad == "Cuenca", 1.02, 1.00))

    salario = base * mult * city_bonus
    salario = np.clip(salario, 480, 5000)

    # Recalibración global para acercar media a 766 manteniendo recortes
    target_mean = 766
    salario = salario * (target_mean / salario.mean())
    salario = np.clip(salario, 480, 5000)
    salario = np.round(salario, 2)

    # ---------------------------
    # EMAIL (42% NA) + algunos sucios (mayúsculas/espacios)
    # ---------------------------
    dominios = np.array(["gmail.com", "outlook.com", "yahoo.com", "hotmail.com", "empresa.ec"])

    slug = (pd.Series(nombre_raw)
            .str.normalize("NFKD")
            .str.encode("ascii", errors="ignore")
            .str.decode("utf-8")
            .str.lower()
            .str.replace(" ", ".", regex=False))

    email = (slug + "@" + pd.Series(rng.choice(dominios, size=n))).to_numpy(dtype=object)

    email = pd.Series(email).astype(str)
    email = email.where(rng.random(n) > 0.10, email.str.upper())
    email = email.where(rng.random(n) > 0.08, " " + email + " ")
    email = email.to_numpy(dtype=object)

    # 42% missing
    email[rng.random(n) < 0.42] = None

    # ---------------------------
    # FECHA REGISTRO (formatos mezclados) + algunos NA
    # ---------------------------
    start = np.datetime64("2023-01-01")
    end = np.datetime64("2026-02-15")
    days_range = (end - start).astype(int)
    fechas = start + rng.integers(0, days_range, size=n).astype("timedelta64[D]")

    fmt_pick = rng.choice(["iso", "dmy", "mdy_text"], size=n, p=[0.55, 0.30, 0.15])
    fecha_registro = []
    for dt, f in zip(fechas, fmt_pick):
        ts = pd.Timestamp(dt)
        if f == "iso":
            fecha_registro.append(ts.strftime("%Y-%m-%d"))
        elif f == "dmy":
            fecha_registro.append(ts.strftime("%d/%m/%Y"))
        else:
            fecha_registro.append(ts.strftime("%b %d, %Y"))  # "Jan 05, 2025"

    fecha_registro = np.array(fecha_registro, dtype=object)
    fecha_registro[rng.random(n) < 0.06] = None

    # ---------------------------
    # IDENTIFICACIÓN (cédula-like con checksum) + 21% con sufijo 001
    # ---------------------------
    def cedula_ec(prov2):
        # 10 dígitos: [prov(2)][3er<6][serial(6)][checksum(1)]
        third = str(rng.integers(0, 6))
        serial = f"{rng.integers(0, 10**6):06d}"
        base9 = prov2 + third + serial  # 9 dígitos
        digits = np.array([int(d) for d in base9])

        # Algoritmo de verificación (cédula ECU)
        coef = np.array([2,1,2,1,2,1,2,1,2])
        prod = digits * coef
        prod = np.where(prod >= 10, prod - 9, prod)
        ver = (10 - (prod.sum() % 10)) % 10
        return base9 + str(ver)

    identificacion = np.array([cedula_ec(prov_code[c]) for c in ciudad], dtype=object)
    mask_001 = rng.random(n) < 0.21
    identificacion = np.where(mask_001, identificacion + "001", identificacion)

    # ---------------------------
    # DATAFRAME FINAL
    # ---------------------------
    personas = pd.DataFrame({
        "identificacion": identificacion,
        "nombre": nombre,
        "edad": edad,
        "altura_cm": altura_cm,
        "peso_lb": peso_lb,
        "sexo": sexo,
        "salario": salario,
        "ciudad": ciudad,
        "fecha_registro": fecha_registro,
        "nivel_educacion": nivel_educacion,
        "email": email
    })

    return personas


# Ejemplo de uso
personas = generar_personas(n=54153, seed=42)

In [ ]:
personas = pd.concat([personas,personas.sample(1231)]).sample(frac = 0.995).reset_index(drop = True)
personas = personas.sort_values("ciudad")

In [10]:
personas.to_csv("personas.csv", index = False, sep = "|")